# 10. Custom input features: wall slip

The GP surrogate models map a set of local input features to the stresses. By default, these are the eight *base features*

| Index | Feature |
|-|-|
| 0, 1, 2 | density $\rho$, mass fluxes $j_x$, $j_y$ |
| 3, 4, 5 | gap height $h$ and its gradients $\partial h/\partial x$, $\partial h/\partial y$ |
| 6, 7 | wall velocities $U$, $V$ |

In addition, any number of *custom features* can be appended. They start at index 8 and are provided as constant fields on the simulation grid.
This way, the constitutive behavior can depend on quantities that are not part of the gap-averaged solution, e.g. a spatially varying wall chemistry or roughness.

Here, we use the slip length $L_s$ at the upper wall as custom feature. As in [tutorial 7](fvm_07_gp_mock.ipynb), the MD simulations are replaced by a `Mock` object, which evaluates the analytical wall stress (see [tutorial 2](fvm_02_stress_sympy.ipynb)) and interprets feature 8 as the slip length at the upper wall.

## Setting up a problem with custom features

Custom features are passed to the `Problem` constructor as `extra_field`, an array of shape `(num_extra_features, Nx + 2, Ny + 2)`. The two additional cells per direction are ghost cells, which we fill periodically here.
The `Database` needs to know the number of custom features as well, so we construct all parts of the problem explicitly instead of using `Problem.from_string`:

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf
from scipy.ndimage import gaussian_filter

from GaPFlow import Problem, Database
from GaPFlow.io import read_yaml_input
from GaPFlow.md import Mock


def build_problem(input_string, extra):
    """Create a problem with custom GP input features.

    extra: array of shape (num_extra_features, Nx, Ny), without ghost cells.
    """
    with io.StringIO(input_string) as ymlfile:
        inp = read_yaml_input(ymlfile)

    # Add periodic ghost cells
    extra_field = np.pad(extra, ((0, 0), (1, 1), (1, 1)), mode='wrap')

    md_runner = Mock(inp['properties'], inp['geometry'], inp['gp'])
    database = Database(md_runner, inp['db'], num_extra_features=extra.shape[0])

    return Problem(options=inp['options'],
                   grid=inp['grid'],
                   numerics=inp['numerics'],
                   prop=inp['properties'],
                   geo=inp['geometry'],
                   fem_solver=inp['fem_solver'],
                   energy_spec=inp['energy_spec'],
                   gp=inp['gp'],
                   database=database,
                   extra_field=extra_field)

Which features enter a GP model is controlled by `active_dims` in the `gp` section of the input. For the pressure, we only use the density (`[0]`).
For the wall shear stress $\tau_{xz}$, we use the density, the mass flux $j_x$ and the slip length (`[0, 1, 8]`).

## 1D: Partially slipping wall

We consider a flat, periodic channel with a sliding lower wall. The upper wall is no-slip in one half of the domain and slips in the other half.

In [ ]:
slip_1d_input = """
options:
    output: data/slip_1d_lj
    write_freq: 100
    use_tstamp: True
grid:
    Lx: 1470.
    Ly: 1.
    Nx: 200
    Ny: 1
    xE: ['P', 'P', 'P']
    xW: ['P', 'P', 'P']
    yS: ['P', 'P', 'P']
    yN: ['P', 'P', 'P']
geometry:
    type: inclined
    hmin: 12.
    hmax: 12.
    U: 0.12
    V: 0.
numerics:
    CFL: 0.5
    adaptive: 1
    tol: 1e-8
    dt: 0.1
    max_it: 5_000
properties:
    shear: 2.15
    bulk: 0.
    EOS: BWR
    T: 1.0
    rho0: 0.8
gp:
    press:
        fix_noise: True
        atol: 1.
        rtol: 0.
        obs_stddev: 2.e-2
        max_steps: 10
        active_dims: [0]           # density
    shear:
        fix_noise: True
        atol: 1.
        rtol: 0.
        obs_stddev: 4.e-3
        max_steps: 10
        active_dims:
            x: [0, 1, 8]           # density, flux, slip length
db:
    init_size: 10
    init_method: lhc
"""

The slip length is a smooth, periodic step function, built from error functions:

In [ ]:
nx = 200
a = 20.

_erf = erf(np.linspace(-a, a, nx // 2))
slip_length = np.concatenate([_erf, -_erf])
slip_length = (1. + np.roll(slip_length, nx // 4)) / 2.

x = np.linspace(0., 1., nx)
plt.plot(x, slip_length)
plt.xlabel('$x/L_x$')
plt.ylabel('Slip length $L_s$');

In [ ]:
problem_1d = build_problem(slip_1d_input, slip_length[None, :, None])
problem_1d.run()

In [ ]:
problem_1d.plot()

## 2D: Checkerboard pattern

In 2D, the wall shear stress $\tau_{yz}$ gets its own GP model with the features density, mass flux $j_y$ and slip length (`[0, 2, 8]`).

In [ ]:
slip_2d_input = """
options:
    output: data/slip_2d_lj
    write_freq: 100
    use_tstamp: True
grid:
    Lx: 1470.
    Ly: 1470.
    Nx: 64
    Ny: 64
    xE: ['P', 'P', 'P']
    xW: ['P', 'P', 'P']
    yS: ['P', 'P', 'P']
    yN: ['P', 'P', 'P']
geometry:
    type: inclined
    hmin: 10.
    hmax: 10.
    U: 0.12
    V: 0.
numerics:
    CFL: 0.5
    adaptive: 1
    tol: 1e-8
    dt: 0.05
    max_it: 5_000
properties:
    shear: 2.15
    bulk: 0.
    EOS: BWR
    T: 1.0
    rho0: 0.8
gp:
    press:
        fix_noise: True
        atol: 1.
        rtol: 0.
        obs_stddev: 2.e-2
        max_steps: 5
        active_dims: [0]           # density
    shear:
        fix_noise: True
        atol: 1.
        rtol: 0.
        obs_stddev: 4.e-3
        max_steps: 5
        active_dims:
            x: [0, 1, 8]           # density, flux_x, slip length
            y: [0, 2, 8]           # density, flux_y, slip length
db:
    init_size: 10
    init_method: lhc
"""

The slip length is a checkerboard pattern, smoothed with a periodic Gaussian filter:

In [ ]:
nx = ny = 64

slip_length = np.zeros((nx, ny))
slip_length[:nx // 2, :ny // 2] = 1.
slip_length[nx // 2:, ny // 2:] = 1.
slip_length = gaussian_filter(slip_length, sigma=5., mode='wrap')

plt.imshow(slip_length.T, origin='lower', extent=(0., 1., 0., 1.))
plt.colorbar(label='Slip length $L_s$')
plt.xlabel('$x/L_x$')
plt.ylabel('$y/L_y$');

In [ ]:
problem_2d = build_problem(slip_2d_input, slip_length[None])
problem_2d.run()

In [ ]:
problem_2d.plot()

## Summary

To add custom features to a simulation:

1. Pass the number of features to the `Database` (`num_extra_features`).
2. Pass the feature values as `extra_field` to the `Problem`, including ghost cells.
3. Select the features for each GP model with `active_dims`, using indices starting at 8.

With real MD simulations instead of the `Mock` object, the MD runner has to translate the custom features into the simulation setup, e.g. into the wall-fluid interaction that produces the desired slip.